# Production-Grade MNIST Training Pipeline

This notebook trains, evaluates, and exports a convolutional neural network for handwritten digit recognition. It is organized like a small production training job: deterministic setup, centralized configuration, reusable functions, explicit validation, callbacks, artifact management, and model export compatible with the Flask app in this repository.

In [1]:
from __future__ import annotations

import json
import logging
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
LOGGER = logging.getLogger("mnist-training")

## 1. Configuration

Keep operational settings in one place so training runs are easy to reproduce, review, and promote into scripts or CI jobs later.

In [2]:
@dataclass(frozen=True)
class TrainingConfig:
    image_height: int = 28
    image_width: int = 28
    channels: int = 1
    num_classes: int = 10
    validation_size: int = 10_000
    batch_size: int = 64
    epochs: int = 20
    learning_rate: float = 1e-3
    l2_penalty: float = 1e-4
    seed: int = 42
    model_dir: Path = Path("artifacts")
    keras_model_path: Path = Path("artifacts/mnist.keras")
    legacy_model_path: Path = Path("mnist.h5")
    metrics_path: Path = Path("artifacts/metrics.json")

    @property
    def input_shape(self) -> Tuple[int, int, int]:
        return (self.image_height, self.image_width, self.channels)


CONFIG = TrainingConfig()
CONFIG.model_dir.mkdir(parents=True, exist_ok=True)
CONFIG

TrainingConfig(image_height=28, image_width=28, channels=1, num_classes=10, validation_size=10000, batch_size=64, epochs=20, learning_rate=0.001, l2_penalty=0.0001, seed=42, model_dir=PosixPath('artifacts'), keras_model_path=PosixPath('artifacts/mnist.keras'), legacy_model_path=PosixPath('mnist.h5'), metrics_path=PosixPath('artifacts/metrics.json'))

In [3]:
def set_global_determinism(seed: int) -> None:
    """Seed common randomness sources used by TensorFlow training."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as exc:  # Some TF/platform combinations do not support this.
        LOGGER.warning("TensorFlow op determinism is unavailable: %s", exc)


set_global_determinism(CONFIG.seed)
LOGGER.info("Using TensorFlow %s", tf.__version__)

## 2. Data Loading and Preprocessing

Use a real validation split from the training set. Keep the test set untouched until the final evaluation so it remains an honest measure of generalization.

In [4]:
def normalize_images(images: np.ndarray, config: TrainingConfig) -> np.ndarray:
    images = images.astype("float32") / 255.0
    return np.expand_dims(images, axis=-1).reshape((-1, *config.input_shape))


def load_mnist_data(config: TrainingConfig):
    (x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

    x_train_full = normalize_images(x_train_full, config)
    x_test = normalize_images(x_test, config)
    y_train_full = keras.utils.to_categorical(y_train_full, config.num_classes)
    y_test = keras.utils.to_categorical(y_test, config.num_classes)

    rng = np.random.default_rng(config.seed)
    indices = rng.permutation(len(x_train_full))
    validation_indices = indices[: config.validation_size]
    training_indices = indices[config.validation_size :]

    x_train = x_train_full[training_indices]
    y_train = y_train_full[training_indices]
    x_val = x_train_full[validation_indices]
    y_val = y_train_full[validation_indices]

    return (x_train, y_train), (x_val, y_val), (x_test, y_test)


(x_train, y_train), (x_val, y_val), (x_test, y_test) = load_mnist_data(CONFIG)
LOGGER.info("Train: %s | Validation: %s | Test: %s", x_train.shape, x_val.shape, x_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
def build_training_dataset(x: np.ndarray, y: np.ndarray, config: TrainingConfig) -> tf.data.Dataset:
    data_augmentation = keras.Sequential(
        [
            layers.RandomRotation(0.03, seed=config.seed),
            layers.RandomZoom(0.08, seed=config.seed),
            layers.RandomTranslation(0.08, 0.08, seed=config.seed),
        ],
        name="data_augmentation",
    )

    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x), seed=config.seed, reshuffle_each_iteration=True)
    dataset = dataset.batch(config.batch_size)
    dataset = dataset.map(lambda images, labels: (data_augmentation(images, training=True), labels),
                          num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.prefetch(tf.data.AUTOTUNE)


def build_eval_dataset(x: np.ndarray, y: np.ndarray, config: TrainingConfig) -> tf.data.Dataset:
    return tf.data.Dataset.from_tensor_slices((x, y)).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = build_training_dataset(x_train, y_train, CONFIG)
val_ds = build_eval_dataset(x_val, y_val, CONFIG)
test_ds = build_eval_dataset(x_test, y_test, CONFIG)

## 3. Model Definition

The model is intentionally compact for MNIST, but includes regularization and batch normalization to keep training stable.

In [6]:
def build_model(config: TrainingConfig) -> keras.Model:
    weight_decay = regularizers.l2(config.l2_penalty)

    inputs = keras.Input(shape=config.input_shape, name="image")
    x = layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=weight_decay)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=weight_decay)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=weight_decay)(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=weight_decay)(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(config.num_classes, activation="softmax", name="digit_probabilities")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="mnist_digit_classifier")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_model(CONFIG)
model.summary()

Model: "mnist_digit_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ digit_probabilities (Dense)     │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,370 (435.04 KB)

 Trainable params: 110,922 (433.29 KB)

 Non-trainable params: 448 (1.75 KB)

## 4. Training

Callbacks protect the run from over-training, save the best artifact, and reduce the learning rate when validation quality plateaus.

In [7]:
def build_callbacks(config: TrainingConfig) -> list[keras.callbacks.Callback]:
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=config.keras_model_path,
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]


history = model.fit(
    train_ds,
    epochs=CONFIG.epochs,
    validation_data=val_ds,
    callbacks=build_callbacks(CONFIG),
)

Epoch 1/20
781/782 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.7715 - loss: 0.7492
Epoch 1: val_accuracy improved from None to 0.97110, saving model to artifacts/mnist.keras

Epoch 1: finished saving model to artifacts/mnist.keras
782/782 ━━━━━━━━━━━━━━━━━━━━ 140s 173ms/step - accuracy: 0.8970 - loss: 0.3753 - val_accuracy: 0.9711 - val_loss: 0.1328 - learning_rate: 0.0010
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.9677 - loss: 0.1515
Epoch 2: val_accuracy improved from 0.97110 to 0.97960, saving model to artifacts/mnist.keras

Epoch 2: finished saving model to artifacts/mnist.keras
782/782 ━━━━━━━━━━━━━━━━━━━━ 127s 163ms/step - accuracy: 0.9709 - loss: 0.1398 - val_accuracy: 0.9796 - val_loss: 0.1056 - learning_rate: 0.0010
Epoch 3/20
781/782 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.9770 - loss: 0.1213
Epoch 3: val_accuracy improved from 0.97960 to 0.98600, saving model to artifacts/mnist.keras

Epoch 3: finished saving model to artifacts/mnist.ke

## 5. Evaluation

Evaluate only once on the test set after training decisions are complete.

In [8]:
def evaluate_model(model: keras.Model, dataset: tf.data.Dataset) -> dict[str, float]:
    loss, accuracy = model.evaluate(dataset, verbose=0)
    return {"test_loss": float(loss), "test_accuracy": float(accuracy)}


def confusion_matrix(model: keras.Model, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    probabilities = model.predict(x, batch_size=CONFIG.batch_size, verbose=0)
    predicted = np.argmax(probabilities, axis=1)
    actual = np.argmax(y, axis=1)
    return tf.math.confusion_matrix(actual, predicted, num_classes=CONFIG.num_classes).numpy()


metrics = evaluate_model(model, test_ds)
cm = confusion_matrix(model, x_test, y_test)

LOGGER.info("Test loss: %.4f", metrics["test_loss"])
LOGGER.info("Test accuracy: %.4f", metrics["test_accuracy"])
print(json.dumps(metrics, indent=2))
print(cm)

{
  "test_loss": 0.04954931139945984,
  "test_accuracy": 0.9936000108718872
}
[[ 977    0    1    0    0    0    1    1    0    0]
 [   0 1131    0    1    0    0    0    3    0    0]
 [   1    0 1028    0    0    0    0    3    0    0]
 [   0    0    0 1008    0    0    0    1    0    1]
 [   0    0    0    0  972    0    0    1    1    8]
 [   1    1    0    5    0  884    1    0    0    0]
 [   4    6    0    0    1    2  945    0    0    0]
 [   0    0    6    0    1    0    0 1021    0    0]
 [   1    0    2    1    1    1    0    0  968    0]
 [   0    0    0    1    2    0    0    4    0 1002]]


## 6. Export Artifacts

Save the modern Keras artifact for training lineage and the legacy `mnist.h5` file expected by the existing Flask app.

In [9]:
def export_artifacts(model: keras.Model, metrics: dict[str, float], config: TrainingConfig) -> None:
    model.save(config.keras_model_path)
    model.save(config.legacy_model_path)

    payload = {
        "config": {key: str(value) if isinstance(value, Path) else value for key, value in asdict(config).items()},
        "metrics": metrics,
        "tensorflow_version": tf.__version__,
    }
    config.metrics_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    LOGGER.info("Saved model: %s", config.keras_model_path)
    LOGGER.info("Saved Flask-compatible model: %s", config.legacy_model_path)
    LOGGER.info("Saved metrics: %s", config.metrics_path)


export_artifacts(model, metrics, CONFIG)

## 7. Inference Contract Check

This mirrors the Flask preprocessing path: grayscale 28x28 image, batch dimension, normalized pixel values, and a probability distribution over digits.

In [10]:
def predict_digit(model: keras.Model, image: np.ndarray) -> dict[str, object]:
    if image.shape != CONFIG.input_shape:
        raise ValueError(f"Expected image shape {CONFIG.input_shape}, got {image.shape}")

    batch = np.expand_dims(image.astype("float32"), axis=0)
    probabilities = model.predict(batch, verbose=0)[0]
    digit = int(np.argmax(probabilities))
    return {
        "digit": digit,
        "confidence": float(probabilities[digit]),
        "probabilities": probabilities.tolist(),
    }


sample_prediction = predict_digit(model, x_test[0])
print(json.dumps(sample_prediction, indent=2))

{
  "digit": 7,
  "confidence": 0.9999932050704956,
  "probabilities": [
    3.076658927625431e-09,
    3.1620695608580718e-06,
    1.3448333220367203e-06,
    4.624443192824401e-07,
    1.4324764663342648e-07,
    7.47977946247147e-09,
    1.2367196676466019e-11,
    0.9999932050704956,
    1.1653988885029776e-08,
    1.5578455077047693e-06
  ]
}


## Production Notes

- Keep the notebook as an experiment and artifact-generation entry point; move shared preprocessing and model-loading code into importable Python modules when the service grows.
- The Flask app should load one model at process startup, validate upload inputs, and reuse the same preprocessing contract shown above.
- Track `artifacts/metrics.json` with each trained model so releases can be compared before deployment.